# 07 · Collection and the held-out gate

Two things this suite could not do before: generate training
trajectories from the model's own attempts, and measure whether
a trained checkpoint is actually better than the stock one.

**This notebook imports the shared code instead of restating
it.** Notebooks 00-06 keep self-contained cells because each one
needs only a handful of definitions. The episode loop, the
collection filters and the scorecard are none of those: three
hand-copied copies would drift, and a gate that drifts from the
collector it grades is worse than no gate. The GPU-specific part
— turning messages into one generated turn — is the only thing
defined here.

**Held-out means held out.** The evaluation families in
`qwen3_8_27b_code.tasks` share no bug class, module or family
name with the SFT fixtures, and the test suite enforces that.
Never collect training data from them.

## Install the pinned day-zero environment

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import gc
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

# IPython stores the last exception on sys.last_traceback, whose frames keep
# every local alive, including a ~52 GiB model from a failed cell. gc.collect()
# cannot free what those frames still reference.
def release_stale_gpu_state() -> float:
    for _stale_name in ("model", "tokenizer", "processor", "trainer"):
        globals().pop(_stale_name, None)
    for _exc_attr in ("last_traceback", "last_value", "last_type", "last_exc"):
        if hasattr(sys, _exc_attr):
            delattr(sys, _exc_attr)
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch._dynamo.reset()
    except AttributeError:
        pass
    return torch.cuda.mem_get_info()[0] / 1024**3

# Fail before a model load that accelerate would silently offload.
def require_free_vram(minimum_gib: float) -> float:
    free_gib = release_stale_gpu_state()
    if free_gib < minimum_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free but this load needs about "
            f"{minimum_gib:.0f} GiB. A previous model in this kernel is still "
            "holding memory. Restart the runtime and rerun from the top."
        )
    return free_gib

# Reject a load that accelerate quietly spilled to CPU or disk. A partially
# offloaded model copies weights back per forward pass (the 2.4 GiB embedding
# alone) and is guaranteed to OOM or crawl mid-episode.
def assert_model_fully_resident(model, minimum_free_gib: float = 4.0) -> None:
    non_cuda = sorted({
        parameter.device.type
        for parameter in model.parameters()
        if parameter.device.type != "cuda"
    })
    offload_hooks = [
        name for name, module in model.named_modules()
        if getattr(getattr(module, "_hf_hook", None), "offload", False)
    ]
    if non_cuda or offload_hooks:
        raise RuntimeError(
            "The checkpoint did not fit on the GPU and accelerate offloaded "
            f"part of it (devices={non_cuda}, offload_hooks={len(offload_hooks)}). "
            "Restart the runtime to release stale VRAM, then rerun from the top."
        )
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    if free_gib < minimum_free_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free after the load; the KV "
            "cache and generation workspaces need headroom. Restart the "
            "runtime and rerun from the top."
        )
    print(f"Model fully resident on GPU; {free_gib:.1f} GiB VRAM free.")

release_stale_gpu_state()

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]


def require_private_repo(repo_id: str, repo_type: str = "model") -> None:
    # Refuse to publish into a Hub repo that already exists and is public.
    # private=True on create_repo, push_to_hub and hub_private_repo applies
    # only when the repo is created; an existing public repo stays public
    # and every later push lands in the open.
    from huggingface_hub import HfApi

    api = HfApi(token=hf_token)
    if not api.repo_exists(repo_id, repo_type=repo_type):
        return
    if not api.repo_info(repo_id, repo_type=repo_type).private:
        raise RuntimeError(
            f"{repo_type} repo {repo_id} exists and is public. Make it private first with "
            f"HfApi(token=hf_token).update_repo_settings(repo_id={repo_id!r}, repo_type={repo_type!r}, "
            "private=True), or publish under a new id."
        )

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Bring in the shared harness, collector and gate

In [ ]:
import subprocess

REPO_URL = "https://github.com/CodeHalwell/qwen3.8-27B-code"
REPO_REVISION = "main"  # Pin an immutable commit before a run that produces artifacts.
REPO_DIR = Path("/content/qwen3.8-27B-code")

import shutil

# Cloning only when the directory was absent meant a rerun in a
# runtime that already held a checkout kept whatever was cloned
# first, package included, so the harness fingerprint would
# describe code this run is not using. Fetch and reset instead;
# a fetch takes a branch or a commit, where --branch takes only
# a branch.
if not (REPO_DIR / ".git").is_dir():
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "init", "-q", str(REPO_DIR)], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "add", "origin", REPO_URL], check=True
    )
subprocess.run(
    ["git", "fetch", "--depth", "1", "origin", REPO_REVISION], cwd=REPO_DIR, check=True
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
# A rerun would otherwise import the copy an earlier run of this
# cell left in sys.modules, refreshed checkout or not.
for module_name in [
    name for name in sys.modules if name.split(".")[0] == "qwen3_8_27b_code"
]:
    del sys.modules[module_name]

from qwen3_8_27b_code.collection import collect, write_corpus
from qwen3_8_27b_code.episodes import EpisodeBudget, TurnResult
from qwen3_8_27b_code.evaluation import (
    DEFAULT_SEEDS,
    build_provenance,
    compare,
    effort_ladder,
    evaluate,
    gate,
    gate_passed,
    pairing_problems,
    provenance_mismatches,
    read_report,
    write_report,
)
from qwen3_8_27b_code.fixtures import iter_tasks
from qwen3_8_27b_code.long_horizon import training_tasks
from qwen3_8_27b_code.tasks import evaluation_tasks, task_from_fixture
from qwen3_8_27b_code.thinking import build_reasoning_length_pairs, write_length_pairs

repo_revision = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True, capture_output=True, check=True,
).stdout.strip()
print(json.dumps({"repo": REPO_URL, "revision": repo_revision}, indent=2))

## Run configuration

In [ ]:
from unsloth import FastModel

MODEL_ID = "unsloth/Qwen3.8-27B"
# A Hub id is mutable; the baseline is loaded at this revision and
# recorded at the commit it resolves to. Pin an immutable commit
# for a run that produces artifacts.
MODEL_REVISION = "main"
ACCEPTED_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
ACCEPTED_REVISION = "main"  # pin a commit to repeat a gate exactly
# Gate the latest stage that finished: notebook 04's adapter when
# it has pushed one, else notebook 03's. False gates
# ACCEPTED_ADAPTER_ID as configured.
DPO_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-dpo-lora"
DPO_REVISION = "main"  # a commit of the DPO repo; ACCEPTED_REVISION pins the SFT repo
GATE_LATEST_STAGE = True
# Every think block after the request stays in context for the rest
# of the episode, so a thirty-call episode carries thirty turns of
# reasoning: this is where thinking and horizon meet, and why the
# window is sized for the long band rather than for one turn.
MAX_SEQUENCE_LENGTH = 32_768
# Reasoning counts against max_new_tokens, and a turn cut off inside
# its think block returns no action. One cap sized for medium turns
# the xhigh rung into a truncation measurement, so each effort gets
# its own; set each above the p95 reasoning length measured at that
# effort (docs/training-plan.md, Stage 0). `high` is deliberately
# absent: the template aliases it to xhigh.
MAX_NEW_TOKENS_BY_EFFORT = {"low": 2_048, "medium": 4_096, "xhigh": 8_192}
REASONING_EFFORT = "medium"
# docs/thinking-budget.md: a candidate may spend at most this
# fraction more reasoning tokens per turn than the baseline.
# Quality is gated first and separately; this only stops a
# "better" checkpoint that got there by thinking longer.
MAX_REASONING_GROWTH = 0.10

# docs/evaluation.md funnel: the sentinel tier is the cheap one
# every candidate runs. Widen only for a candidate or release
# gate, and price it before starting.
EVAL_VARIANTS_PER_FAMILY = 1     # 9 held-out tasks: 6 short, 2 medium, 1 long
EVAL_ATTEMPTS = 2                # two seeds per task; a candidate must match its baseline
# Ceilings, not targets: the long band runs to 30 tool calls and a
# smaller budget excludes the pipeline tasks by construction.
EPISODE_BUDGET = EpisodeBudget(tool_calls=30, wall_seconds=900.0)

RUN_BASELINE_EVAL = None         # None: only when no baseline could be pulled from the Hub
# The stock model at low, medium and xhigh, each with its own cap;
# picks the deployment effort and the gate baseline
# (docs/thinking-budget.md, lever 1).
RUN_EFFORT_LADDER = False
EFFORT_LADDER_TOLERANCE = 0.0    # a rung must match the best success to be eligible
RUN_CANDIDATE_EVAL = None        # None: when the adapter notebook 03 pushes exists on the Hub
RUN_COLLECTION = False           # expensive; read the cost note below first
PUSH_ARTIFACTS = True

COLLECTION_ATTEMPTS = 3
COLLECTION_VARIANTS_PER_FAMILY = 2
COLLECTION_SEEDS = (3407, 9176, 20261)

REPORT_DIR = RUN_ROOT / "gate"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Colab runtimes are per notebook and per session, so a baseline
# measured today is gone before the candidate exists. The reports
# live in a private dataset repo; the last cell pushes this
# session's REPORT_DIR and this pulls the earlier ones into a
# separate directory. Nothing pulled is ever mistaken for this
# session's work: the gate takes the baseline by name from
# either place and the candidate only from this session.
GATE_REPORTS_REPO = f"{HF_USERNAME}/qwen38-code-gate-reports"
PULL_REPORTS_FROM_HUB = True
HUB_REPORT_DIR = RUN_ROOT / "gate_hub"
# The frozen baseline the gate pairs with this session's candidate:
# baseline.json, or a ladder rung such as ladder_medium.json.
GATE_BASELINE_FILE = "baseline.json"
if PULL_REPORTS_FROM_HUB:
    from huggingface_hub import HfApi, snapshot_download

    if HfApi(token=hf_token).repo_exists(GATE_REPORTS_REPO, repo_type="dataset"):
        snapshot_download(
            GATE_REPORTS_REPO,
            repo_type="dataset",
            local_dir=str(HUB_REPORT_DIR),
            allow_patterns=["*.json", "*.jsonl"],
            token=hf_token,
        )
        print(f"pulled earlier reports: {sorted(p.name for p in HUB_REPORT_DIR.iterdir())}")
    else:
        print(f"no earlier reports at {GATE_REPORTS_REPO}; starting fresh.")

from huggingface_hub import HfApi

def resolved_revision(repo_id: str, revision: str) -> str:
    # The commit a branch or tag points at now, so the record
    # names the checkpoint that was measured, not a moving ref.
    return HfApi(token=hf_token).model_info(repo_id, revision=revision).sha

stock_model_ref = f"{MODEL_ID}@{resolved_revision(MODEL_ID, MODEL_REVISION)}"
print(json.dumps({"stock_model": stock_model_ref}, indent=2))

# Recorded on every report this notebook writes, with the same
# writer the CLI uses. The gate refuses to pair two reports
# whose settings differ, and records a harness revision that does.
def report_provenance(model_ref: str, reasoning_effort: str = REASONING_EFFORT) -> dict:
    return build_provenance(
        model=model_ref,
        harness_revision=repo_revision,
        reasoning_effort=reasoning_effort,
        max_new_tokens=MAX_NEW_TOKENS_BY_EFFORT[reasoning_effort],
        max_sequence_length=MAX_SEQUENCE_LENGTH,
        episode_budget=EPISODE_BUDGET,
        attempts_per_task=EVAL_ATTEMPTS,
        seeds=DEFAULT_SEEDS[:EVAL_ATTEMPTS],
        variants_per_family=EVAL_VARIANTS_PER_FAMILY,
    )

# Publishing is the last cell, but an existing public target
# is found now, before any GPU time is spent.
if PUSH_ARTIFACTS:
    require_private_repo(GATE_REPORTS_REPO, "dataset")

# What this session does follows from what already exists: a
# baseline is measured once and reused; a candidate is gated as
# soon as notebook 03 has pushed an adapter.
from huggingface_hub import HfApi

if RUN_BASELINE_EVAL is None:
    pulled_baseline = HUB_REPORT_DIR / GATE_BASELINE_FILE
    RUN_BASELINE_EVAL = True
    if pulled_baseline.exists():
        # Reuse it only if it was measured the way this session
        # measures: otherwise the candidate would be evaluated
        # in full and then refused at the gate.
        mismatches = provenance_mismatches(
            read_report(pulled_baseline).metadata, report_provenance(stock_model_ref)
        )
        RUN_BASELINE_EVAL = bool(mismatches)
        for line in mismatches:
            print(f"pulled baseline differs, measuring a new one: {line}")
if not RUN_BASELINE_EVAL:
    # The gate prefers this session's file over the pulled copy,
    # so one left by an earlier run in this runtime must go.
    (REPORT_DIR / GATE_BASELINE_FILE).unlink(missing_ok=True)
# The candidate revision is pinned here, before the evaluation,
# so a push during it cannot change what the report and the
# merge name. Notebook 03 removes its run manifest when training
# starts and uploads it after the final adapter push, so a
# revision that carries it is a training run that finished.
api = HfApi(token=hf_token)
CANDIDATE_REVISION = None
candidates = (
    ((DPO_ADAPTER_ID, DPO_REVISION), (ACCEPTED_ADAPTER_ID, ACCEPTED_REVISION))
    if GATE_LATEST_STAGE else ((ACCEPTED_ADAPTER_ID, ACCEPTED_REVISION),)
)
for adapter_id, pinned in candidates:
    if not api.repo_exists(adapter_id):
        continue
    revision = resolved_revision(adapter_id, pinned)
    if api.file_exists(adapter_id, "run_manifest.json", revision=revision):
        ACCEPTED_ADAPTER_ID, CANDIDATE_REVISION = adapter_id, revision
        break
if RUN_CANDIDATE_EVAL is None:
    RUN_CANDIDATE_EVAL = CANDIDATE_REVISION is not None
if RUN_CANDIDATE_EVAL and CANDIDATE_REVISION is None:
    raise RuntimeError("No finished adapter to gate; run notebook 03 to completion first.")
print(json.dumps({"run_baseline_eval": RUN_BASELINE_EVAL, "run_candidate_eval": RUN_CANDIDATE_EVAL}, indent=2))

# Six single-file families (short band) plus the three multi-file
# families from long_horizon: two coupled-module (medium band) and
# one four-stage pipeline (long band).
evaluation_suite = evaluation_tasks(variants_per_family=EVAL_VARIANTS_PER_FAMILY)
print(json.dumps({
    "held_out_tasks": len(evaluation_suite),
    "families": sorted({task.family for task in evaluation_suite}),
    "multi_file_tasks": sum(len(task.gold_files) > 1 for task in evaluation_suite),
    "attempts_each": EVAL_ATTEMPTS,
}, indent=2))

## The only GPU-specific piece: messages in, one turn out

In [ ]:
# Everything else in this notebook is shared code. A policy is a
# callable that renders the history, generates one assistant
# turn, and reports whether generation finished or was cut off.
def build_policy_factory(model, tokenizer, reasoning_effort=REASONING_EFFORT):
    max_new_tokens = MAX_NEW_TOKENS_BY_EFFORT[reasoning_effort]
    text_tokenizer = text_tokenizer_of(tokenizer)
    generation_eos = model.generation_config.eos_token_id
    eos_token_ids = {
        token_id
        for token_id in (
            *(generation_eos if isinstance(generation_eos, (list, tuple)) else [generation_eos]),
            text_tokenizer.eos_token_id,
        )
        if token_id is not None
    }
    if not eos_token_ids:
        raise RuntimeError("No end-of-turn token id is available; truncation cannot be detected.")
    # The thinking budget is measured in tokens generated before
    # the think block closes. Count them from the ids, not from
    # decoded text, so the number is exact.
    think_end_id = text_tokenizer.convert_tokens_to_ids("</think>")
    if think_end_id is None or think_end_id == text_tokenizer.unk_token_id:
        raise RuntimeError("The tokenizer has no </think> token; reasoning tokens cannot be counted.")

    def policy_factory(task, seed):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        def policy(messages):
            rendered = render_chat(
                messages,
                add_generation_prompt=True,
                reasoning_effort=reasoning_effort,
            )
            inputs = tokenizer(
                text=rendered, return_tensors="pt", add_special_tokens=False
            ).to("cuda")
            prompt_tokens = int(inputs["input_ids"].numel())
            if prompt_tokens + max_new_tokens > MAX_SEQUENCE_LENGTH:
                return TurnResult(
                    text="", prompt_tokens=prompt_tokens, fault="context_budget"
                )
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=1.0,
                    top_p=0.95,
                    top_k=20,
                    do_sample=True,
                    use_cache=True,
                )
            new_ids = outputs[0, inputs["input_ids"].shape[1]:]
            completion_tokens = int(new_ids.numel())
            stopped_on_eos = completion_tokens > 0 and int(new_ids[-1]) in eos_token_ids
            text = tokenizer.decode(new_ids, skip_special_tokens=False)
            closes = (new_ids == think_end_id).nonzero()
            if len(closes):
                # Everything up to and including </think> was reasoning.
                reasoning_tokens = int(closes[0].item()) + 1
            elif rendered.rstrip().endswith("<think>") or "<think>" in text:
                # The block never closed: the whole turn was
                # reasoning, which is the overrun the budget
                # has to see rather than hide.
                reasoning_tokens = completion_tokens
            else:
                reasoning_tokens = 0
            return TurnResult(
                text=text,
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                fault=None if stopped_on_eos else "output_truncated",
                reasoning_tokens=reasoning_tokens,
            )

        return policy

    return policy_factory

In [ ]:
# Bumped from v1 when the `shell` description stopped carrying pilot status
# text. Tool descriptions are model inputs and part of the fingerprint, so a
# wording change is a schema change.
TOOL_SCHEMA_VERSION = "qwen38-six-tools-v3"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run one bash command in the repository; output is bounded and a non-zero exit code is reported.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def unify_columns(rows: list[dict]) -> list[dict]:
    """Give every row every key that any row in the list carries.

    ``Dataset.from_list`` names its columns from the first row alone, so a
    key that row happens not to carry is dropped from the whole table: a
    corpus whose bootstrap rows predate ``lane`` silently loses the lane of
    every public row after them, and non-agentic rows are then read as
    agentic. Filling the gaps with None keeps each row's own value and
    leaves the absent ones null, which is what the readers already expect.
    """
    columns = sorted({key for row in rows for key in row})
    return [{column: row.get(column) for column in columns} for row in rows]

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def rendered_tool_schema(rendered_prompt: str) -> str:
    """Extract and canonicalise JSON tool declarations from a Qwen prompt."""
    start_tag = "<tools>"
    end_tag = "</tools>"
    if start_tag not in rendered_prompt or end_tag not in rendered_prompt:
        raise ValueError("Rendered prompt does not contain a <tools> block.")
    payload = rendered_prompt.split(start_tag, 1)[1].split(end_tag, 1)[0]
    try:
        rendered_tools = [
            json.loads(line)
            for line in payload.splitlines()
            if line.strip()
        ]
    except json.JSONDecodeError as exc:
        raise ValueError("Rendered <tools> block is not newline-delimited JSON.") from exc
    return canonical_tool_schema(rendered_tools)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Merge the leading policy messages into one system message.

    The Qwen3.8 template accepts `developer` natively and merges a run of
    leading system/developer messages itself. This fold is therefore not a
    compatibility shim: it exists so training and deployment both hand the
    template one deterministically joined policy message.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def text_tokenizer_of(tokenizer):
    """The text tokenizer behind a multimodal processor, or the tokenizer itself.

    FastModel hands back a processor for this vision-capable checkpoint. It
    renders chat templates and decodes, but a bare positional string is read
    as an image and token-level attributes (eos, added tokens) live one
    level down. Reach through for those, and pass text= otherwise.
    """
    return getattr(tokenizer, "tokenizer", tokenizer)


def render_chat(messages: list[dict], *, add_generation_prompt: bool, reasoning_effort: str = "medium") -> str:
    return text_tokenizer_of(tokenizer).apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort=reasoning_effort,
        preserve_thinking=True,
    )

## Baseline: the stock model on the held-out suite

In [ ]:
baseline_report_path = REPORT_DIR / "baseline.json"

if RUN_BASELINE_EVAL:
    # A baseline from an earlier run of this cell must not
    # survive an evaluation that fails before it writes.
    baseline_report_path.unlink(missing_ok=True)
    require_free_vram(60.0)
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        revision=MODEL_REVISION,
        max_seq_length=MAX_SEQUENCE_LENGTH,
        load_in_4bit=False,
        full_finetuning=False,
        token=hf_token,
    )
    assert_model_fully_resident(model)
    FastModel.for_inference(model)

    baseline = evaluate(
        evaluation_suite,
        build_policy_factory(model, tokenizer),
        label="upstream-bf16",
        attempts_per_task=EVAL_ATTEMPTS,
        budget=EPISODE_BUDGET,
    )
    baseline.metadata = report_provenance(stock_model_ref)
    write_report(baseline, baseline_report_path)
    print(json.dumps(baseline.scorecard(), indent=2))
    print(f"wrote {baseline_report_path}")
else:
    print("Baseline evaluation is off. It is the comparison point for every later claim.")

## Effort ladder: the stock model at low, medium and xhigh

Lever 1 of docs/thinking-budget.md, and it costs no training.
Each rung runs with its own per-turn cap, so the table
measures effort rather than truncation. The recommendation
is the rung that thinks least among those that keep the best
success; set `REASONING_EFFORT` to it and use its report as
the gate baseline. Read `success_by_task_horizon` before the
aggregate: a rung that holds the short tasks and loses the
pipeline is not a cheaper rung.

In [ ]:
if RUN_EFFORT_LADDER:
    if "model" not in globals():
        raise RuntimeError("Load the stock model in the baseline cell first.")
    ladder_reports = {}
    for effort in ("low", "medium", "xhigh"):
        ladder_reports[effort] = evaluate(
            evaluation_suite,
            build_policy_factory(model, tokenizer, reasoning_effort=effort),
            label=f"upstream-bf16-{effort}",
            attempts_per_task=EVAL_ATTEMPTS,
            budget=EPISODE_BUDGET,
        )
        ladder_reports[effort].metadata = report_provenance(stock_model_ref, reasoning_effort=effort)
        write_report(ladder_reports[effort], REPORT_DIR / f"ladder_{effort}.json")
    ladder = effort_ladder(ladder_reports, success_tolerance=EFFORT_LADDER_TOLERANCE)
    (REPORT_DIR / "effort_ladder.json").write_text(json.dumps(ladder, indent=2))
    print(json.dumps(ladder, indent=2))
    if ladder["recommended"] is None:
        print(f"{ladder['note']}. Do not set REASONING_EFFORT from this ladder.")
    else:
        print(
            f"Recommended deployment effort: {ladder['recommended']}. Set REASONING_EFFORT to it "
            f"and use ladder_{ladder['recommended']}.json as the frozen baseline for the gate."
        )
else:
    print("Effort ladder is off. Run it once on the stock model before choosing REASONING_EFFORT.")

## Candidate: the accepted adapter on the same frozen suite

Release the baseline model first. Two 27B checkpoints do not
coexist on one card, and a partially offloaded second load
crawls or dies mid-episode.

In [ ]:
candidate_report_path = REPORT_DIR / "candidate.json"
# Set only when this cell writes the report; the gate reads it
# rather than inferring from the flag and a file that may be
# left over from an earlier run in the same runtime.
candidate_written = False

if RUN_CANDIDATE_EVAL:
    candidate_report_path.unlink(missing_ok=True)
    release_stale_gpu_state()
    require_free_vram(60.0)
    model, tokenizer = FastModel.from_pretrained(
        model_name=ACCEPTED_ADAPTER_ID,
        revision=CANDIDATE_REVISION,
        max_seq_length=MAX_SEQUENCE_LENGTH,
        load_in_4bit=False,
        token=hf_token,
    )
    assert_model_fully_resident(model)
    FastModel.for_inference(model)

    candidate = evaluate(
        evaluation_suite,
        build_policy_factory(model, tokenizer),
        label="sft-lora-candidate",
        attempts_per_task=EVAL_ATTEMPTS,
        budget=EPISODE_BUDGET,
    )
    candidate_model_ref = f"{ACCEPTED_ADAPTER_ID}@{CANDIDATE_REVISION}"
    candidate.metadata = report_provenance(candidate_model_ref)
    write_report(candidate, candidate_report_path)
    candidate_written = True
    print(json.dumps(candidate.scorecard(), indent=2))
else:
    print("Candidate evaluation is off. Turn it on once an adapter revision is accepted.")

## Apply the gate

In [ ]:
# The candidate is always the one measured in this session. The
# baseline is GATE_BASELINE_FILE from this session if it wrote
# one, else the pulled copy. A pulled candidate is never used:
# a fresh baseline gated against a stale candidate would
# republish a verdict nobody asked for.
comparison_path = REPORT_DIR / "comparison.json"
# A verdict from an earlier run of this cell in the same runtime
# must not survive a gate that is skipped or refused now, or the
# persist cell would push it as if it were this run's.
comparison_path.unlink(missing_ok=True)
# The same goes for an acceptance: it is written only by a gate
# that passed in this run.
accepted_path = REPORT_DIR / "accepted.json"
accepted_path.unlink(missing_ok=True)
# The baseline is, in order: the named file this session wrote (a
# ladder rung), the baseline this session measured, then the pulled
# copy of the named file. A fresh measurement always outranks the
# pulled copy it was measured to replace.
baseline_candidates = [REPORT_DIR / GATE_BASELINE_FILE]
if RUN_BASELINE_EVAL:
    baseline_candidates.append(baseline_report_path)
baseline_candidates.append(HUB_REPORT_DIR / GATE_BASELINE_FILE)
baseline_for_gate = next((path for path in baseline_candidates if path.exists()), None)
if not (RUN_CANDIDATE_EVAL and globals().get("candidate_written") and candidate_report_path.exists()):
    print("The gate needs a candidate measured in this session (RUN_CANDIDATE_EVAL).")
elif baseline_for_gate is None:
    print(f"No {GATE_BASELINE_FILE} in this session or on {GATE_REPORTS_REPO}; measure a baseline first.")
elif (blocking := pairing_problems(
    baseline_report := read_report(baseline_for_gate),
    candidate_report := read_report(candidate_report_path),
))[0]:
    for problem in blocking[0]:
        print(f"  [REFUSED] {problem}")
    print("GATE NOT RUN: the two reports were not measured the same way.")
else:
    advisory = blocking[1]
    for note in advisory:
        print(f"  [NOTE] {note}")
    comparison = compare(baseline_report, candidate_report)
    comparison["provenance"] = {
        "baseline": {**baseline_report.metadata, "path": str(baseline_for_gate)},
        "candidate": candidate_report.metadata,
        "notes": advisory,
    }
    checks = gate(comparison, max_reasoning_growth=MAX_REASONING_GROWTH)
    comparison["gate"] = [
        {"name": check.name, "passed": check.passed, "detail": check.detail}
        for check in checks
    ]
    comparison["gate_passed"] = gate_passed(checks)
    comparison_path.write_text(json.dumps(comparison, indent=2))

    print(json.dumps(comparison["deltas"], indent=2))
    # Reasoning tokens per turn, share of generation spent thinking,
    # and success by horizon band, baseline against candidate.
    print(json.dumps(comparison["thinking"], indent=2))
    for check in checks:
        print(f"  [{'PASS' if check.passed else 'FAIL'}] {check.name}: {check.detail}")
    task_level = comparison["task_level"]
    # A suite this small reports paired outcomes; it cannot
    # support a percentage-point significance claim.
    print(
        f"{task_level['wins']} improved, {task_level['losses']} regressed, "
        f"{task_level['ties']} unchanged, of {task_level['tasks']} tasks."
    )
    print("GATE PASSED" if comparison["gate_passed"] else "GATE FAILED")

    if comparison["gate_passed"]:
        # The acceptance record, pushed with the reports: which
        # adapter, at which commit, passed against which baseline.
        accepted_path.write_text(json.dumps({
            "adapter": candidate_model_ref,
            "baseline": baseline_for_gate.name,
            "harness_revision": repo_revision,
            "provenance": candidate_report.metadata,
        }, indent=2))

## Collect training trajectories by rejection sampling

This is the route off the scripted bootstrap corpus. The model
attempts each training task several times, every attempt is
graded from outside its workspace, and only verified attempts
become rows — carrying the model's own reasoning at the effort
it ran at, which is what the scripted corpus cannot supply.

Cost first: attempts × tasks × mean episode seconds. Measure one
task before enabling the full sweep, and use the acceptance rate
in the report to decide whether more attempts or easier tasks
are the better next move.

In [ ]:
if RUN_COLLECTION:
    if "model" not in globals():
        raise RuntimeError("Load a model in one of the cells above before collecting.")
    # Single-file fixtures plus the multi-file training families,
    # so the corpus carries the medium-horizon shape as well.
    collection_tasks = [
        task_from_fixture(fixture)
        for fixture in iter_tasks(COLLECTION_VARIANTS_PER_FAMILY)
    ] + training_tasks(COLLECTION_VARIANTS_PER_FAMILY)
    result = collect(
        collection_tasks,
        build_policy_factory(model, tokenizer),
        attempts_per_task=COLLECTION_ATTEMPTS,
        seeds=COLLECTION_SEEDS,
        budget=EPISODE_BUDGET,
        reasoning_effort=REASONING_EFFORT,
        max_rows_per_task=2,
        # Of the attempts that verified, keep the ones that
        # thought least: the model's own shortest working path.
        selection="shortest_reasoning",
    )
    corpus_path = REPORT_DIR / "collected_trajectories.jsonl"
    report = write_corpus(result, corpus_path, REPORT_DIR / "collection_report.json")
    print(json.dumps(report, indent=2))
    print(f"wrote {corpus_path}; feed it to notebook 02 as SOURCE_LOCAL_JSONL.")

    # Verified attempts that thought more than another verified
    # attempt at the same action become brevity preferences.
    length_pairs = build_reasoning_length_pairs(result.attempts)
    pairs_report = write_length_pairs(
        length_pairs,
        REPORT_DIR / "length_pairs.jsonl",
        REPORT_DIR / "length_pairs_report.json",
    )
    print(json.dumps(pairs_report, indent=2))
    print(
        f"wrote {len(length_pairs)} reasoning-length pairs; feed length_pairs.jsonl to "
        "notebook 04 as LENGTH_PAIRS_LOCAL_JSONL next to the execution-derived pairs."
    )
else:
    print("Collection is off. Enable it once the baseline scorecard shows the failure mix.")

## Persist the reports

Everything this session wrote under `REPORT_DIR` — baseline,
candidate, comparison, ladder rungs, any collected corpus and
its length pairs — goes to one private dataset repo, tagged
with the harness revision that produced it. Pulled copies live
in `HUB_REPORT_DIR` and are not pushed back. The configuration
cell pulls the same repo at the start of the next session.

In [ ]:
if PUSH_ARTIFACTS:
    from huggingface_hub import HfApi

    require_private_repo(GATE_REPORTS_REPO, "dataset")
    api = HfApi(token=hf_token)
    api.create_repo(GATE_REPORTS_REPO, repo_type="dataset", private=True, exist_ok=True)
    commit = api.upload_folder(
        repo_id=GATE_REPORTS_REPO,
        repo_type="dataset",
        folder_path=str(REPORT_DIR),
        allow_patterns=["*.json", "*.jsonl"],
        # An earlier verdict or acceptance on the Hub must not
        # outlive a gate that was skipped, refused or failed here:
        # the remote file is deleted unless this session's copy
        # replaces it (a file uploaded in the same commit is kept).
        delete_patterns=["comparison.json", "accepted.json"],
        commit_message=f"gate reports from {repo_revision[:12]}",
    )
    print(f"pushed {sorted(p.name for p in REPORT_DIR.iterdir())} to {GATE_REPORTS_REPO}")
    print(commit)
else:
    print("PUSH_ARTIFACTS is off; the reports stay in this runtime and vanish with it.")

## What the numbers mean

Read the acceptance rate and the rejection breakdown before the
row count. A corpus of 500 rows whose rejections are dominated
by `completed_without_verification` is telling you the policy
does not verify, and training on the survivors will not fix that.

The difficulty bands come from docs/data-strategy.md: tasks in
the trivial band are protocol smoke tests, the learnable band is
the useful curriculum, and frontier tasks are for later.

The `thinking` section of the report splits reasoning per turn
by outcome. If the attempts that failed thought far more than
the ones that verified, the model is spending tokens on tasks
it cannot do and a tighter budget costs little; if the reverse,
brevity is being bought with correctness and the thinking gate
in the comparison above is the thing to watch.

Feed the collected JSONL to notebook 02, which remains the
publisher that validates, splits and pushes the dataset that
notebooks 03 and 06 consume. The reasoning-length pairs go to
notebook 04 as `LENGTH_PAIRS_LOCAL_JSONL`, where they are
capped to a minority of the mixture (see docs/thinking-budget.md).

Two long-horizon numbers are on every scorecard now:
`peak_prompt_tokens_max`, the largest context any turn needed,
and `context_budget_rate`, how often an episode ran out of
window. When either climbs towards `MAX_SEQUENCE_LENGTH` the
next lever is less thinking per turn, then observation
compaction, in that order.